# Auric CV: train + evaluate one run on Kaggle (end to end)

Designed for **Save Version → Save & Run All (Commit)**, i.e. background mode: every cell runs in order, nothing
needs a click. Before saving:
1. **Add Data**: the dataset holding `cv_dataset.zip` (or its extracted folder with `classmap.txt`).
2. **Add-ons → Secrets**: `GH_TOKEN` = a GitHub token with push access to the repo; attach it to this notebook.
3. **Settings**: Accelerator = GPU, Internet = On.
4. Set `RUN` below.

Paths (from `scripts/_env.sh`): data `/kaggle/tmp/data`, runs `/kaggle/working/runs` (saved as notebook output),
work `/kaggle/tmp/work`. The token is read from Kaggle Secrets and never printed. Results (CSV/JSON/PNG, no
weights) are pushed to GitHub at the end; weights stay in the notebook output under `/kaggle/working/runs`.

In [ ]:
# ---- edit these ----
RUN = "b1"             # "b0" or "b1": which scripts/run_<RUN>.sh to run
RUN_ANALYSIS = True    # also run scripts/run_analysis.sh (errors.py + gt_box_oracle.py) after eval
REPO_URL = "https://github.com/arukh281/auric_cv_assignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/tmp/repo"
RUN_NAMES = {"b0": "b0_full640", "b1": "b1_tile1024"}

In [ ]:
# clone (or update) the repo; the token is used inline and never printed or stored in .git/config
import os, subprocess
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
url = REPO_URL.replace("https://", f"https://{tok}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("git clone failed: " + r.stderr.replace(tok, "***"))
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
del tok, url
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# setup: git pull, install with pinned torch/numpy/opencv constraints, dataset to /kaggle/tmp/data, env lock, all tests
!bash scripts/kaggle_setup.sh

In [ ]:
# train + eval (+ prediction review, merge sensitivity, checkpoint curve), log in /kaggle/working/runs/<run>/run.log
!bash scripts/run_{RUN}.sh

In [ ]:
if RUN_ANALYSIS:
    !bash scripts/run_analysis.sh {RUN_NAMES[RUN]}

In [ ]:
# copy CSV/JSON/PNG (no weights, nothing over 20 MB) into results/ and figures/, commit, push with the GH_TOKEN secret
!bash scripts/kaggle_push_results.sh {RUN_NAMES[RUN]}

In [ ]:
!bash scripts/status.sh {RUN_NAMES[RUN]}